In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

def prepare_isot_dataset():
    """
    Prepares the ISOT Fake News Dataset with CORRECT labels
    Label 0 = REAL NEWS
    Label 1 = FAKE NEWS
    """
    
    os.makedirs('./data', exist_ok=True)
    
    print("=" * 70)
    print("FAKE NEWS DATASET PREPARATION - CORRECTED VERSION")
    print("=" * 70)
    
    fake_path = '/home/koustab/Downloads/data/Fake.csv'
    true_path = '/home/koustab/Downloads/data/True.csv'
    
    # Check if files exist
    if not os.path.exists(fake_path) or not os.path.exists(true_path):
        print(f"\n❌ Error: Dataset files not found!")
        print("\n📥 Please download the ISOT Fake News Dataset:")
        print("1. Go to: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset")
        print("2. Download and extract the ZIP file")
        print("3. Place Fake.csv and True.csv in the ./data/ folder")
        print("\nExpected structure:")
        print("  fake-news-detector/")
        print("  ├── data/")
        print("  │   ├── Fake.csv")
        print("  │   └── True.csv")
        return False
    
    # Load datasets
    print("\n📥 Loading datasets...")
    fake_df = pd.read_csv(fake_path)
    true_df = pd.read_csv(true_path)
    
    print(f"✅ Loaded {len(fake_df)} fake news articles")
    print(f"✅ Loaded {len(true_df)} real news articles")
    print(f"\nColumns: {fake_df.columns.tolist()}")
    
    # Combine title and text if both exist
    print("\n🔧 Processing text columns...")
    if 'title' in fake_df.columns and 'text' in fake_df.columns:
        fake_df['full_text'] = fake_df['title'].fillna('') + ' ' + fake_df['text'].fillna('')
        true_df['full_text'] = true_df['title'].fillna('') + ' ' + true_df['text'].fillna('')
    elif 'text' in fake_df.columns:
        fake_df['full_text'] = fake_df['text'].fillna('')
        true_df['full_text'] = true_df['text'].fillna('')
    else:
        print("❌ Error: No 'text' column found!")
        return False
    
    # Add CORRECT labels
    print("\n🏷️  Adding labels (IMPORTANT: 0=REAL, 1=FAKE)...")
    fake_df['label'] = 1  # FAKE = 1
    true_df['label'] = 0  # REAL = 0
    
    # Keep only necessary columns
    fake_df = fake_df[['full_text', 'label']].rename(columns={'full_text': 'text'})
    true_df = true_df[['full_text', 'label']].rename(columns={'full_text': 'text'})
    
    # Combine datasets
    print("🔄 Combining datasets...")
    df = pd.concat([fake_df, true_df], ignore_index=True)
    
    # Clean data
    print("🧹 Cleaning data...")
    original_len = len(df)
    df = df.dropna()
    df = df[df['text'].str.strip() != '']
    df = df[df['text'].str.len() > 50]  # Remove very short texts
    print(f"Removed {original_len - len(df)} invalid entries")
    
    # Shuffle
    print("🔀 Shuffling dataset...")
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Save prepared dataset
    output_path = './data/news_dataset.csv'
    print(f"\n💾 Saving to {output_path}...")
    df.to_csv(output_path, index=False)
    
    # Verify labels are correct
    print("\n" + "=" * 70)
    print("📊 DATASET STATISTICS")
    print("=" * 70)
    print(f"Total articles: {len(df)}")
    print(f"Real news (label=0): {len(df[df['label']==0])} ({len(df[df['label']==0])/len(df)*100:.1f}%)")
    print(f"Fake news (label=1): {len(df[df['label']==1])} ({len(df[df['label']==1])/len(df)*100:.1f}%)")
    
    # Show sample texts to verify labels
    print("\n" + "=" * 70)
    print("✅ SAMPLE REAL NEWS (label=0):")
    print("=" * 70)
    real_sample = df[df['label']==0].iloc[0]['text'][:300]
    print(real_sample + "...")
    
    print("\n" + "=" * 70)
    print("⚠️  SAMPLE FAKE NEWS (label=1):")
    print("=" * 70)
    fake_sample = df[df['label']==1].iloc[0]['text'][:300]
    print(fake_sample + "...")
    
    print("\n" + "=" * 70)
    print("✅ DATASET PREPARATION COMPLETE!")
    print("=" * 70)
    print(f"\n💾 Dataset saved to: {output_path}")
    print(f"📊 Total samples: {len(df)}")
    print("\n📝 Next steps:")
    print("  1. Verify the samples above look correct")
    print("  2. Run: python train_model.py")
    print("  3. Wait for training (~30-45 minutes)")
    print("  4. Run: python app.py")
    print("=" * 70)
    
    return True

if __name__ == "__main__":
    success = prepare_isot_dataset()
    if not success:
        print("\n❌ Dataset preparation failed. Please check the error messages above.")
    else:
        print("\n✅ Ready to train! Run: python train_model.py")

FAKE NEWS DATASET PREPARATION - CORRECTED VERSION

📥 Loading datasets...
✅ Loaded 23481 fake news articles
✅ Loaded 21417 real news articles

Columns: ['title', 'text', 'subject', 'date']

🔧 Processing text columns...

🏷️  Adding labels (IMPORTANT: 0=REAL, 1=FAKE)...
🔄 Combining datasets...
🧹 Cleaning data...
Removed 9 invalid entries
🔀 Shuffling dataset...

💾 Saving to ./data/news_dataset.csv...

📊 DATASET STATISTICS
Total articles: 44889
Real news (label=0): 21416 (47.7%)
Fake news (label=1): 23473 (52.3%)

✅ SAMPLE REAL NEWS (label=0):
Trump says U.S. upholds and sticks to 'one China' policy: Xinhua BEIJING (Reuters) - The United States government upholds and sticks to the  one China  policy, U.S. President Donald Trump told Chinese President Xi Jinping on Thursday during talks in Beijing, China s official Xinhua news agency repor...

⚠️  SAMPLE FAKE NEWS (label=1):
SHERIFF CLARKE OUTRAGED AT RALLY VIOLENCE: ” Where’s the FBI and DOJ?” Sheriff Clarke weighs in on the violence at the

In [4]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizer, 
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import json

# Configuration
CONFIG = {
    'model_name': 'distilbert-base-uncased',
    'max_length': 512,
    'batch_size': 16,
    'epochs': 1,
    'learning_rate': 2e-5,
    'warmup_steps': 500,
    'save_path': './fake_news_model',
    'dataset_path': './data/news_dataset.csv',
    'test_size': 0.2,
}

class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def load_dataset():
    """Load and split the prepared dataset"""
    print("📥 Loading dataset...")
    
    if not os.path.exists(CONFIG['dataset_path']):
        print(f"❌ Error: {CONFIG['dataset_path']} not found!")
        print("Please run: python prepare_data.py first")
        return None, None, None, None
    
    df = pd.read_csv(CONFIG['dataset_path'])
    
    print(f"\n📊 Dataset loaded:")
    print(f"  Total samples: {len(df)}")
    print(f"  Real news (0): {len(df[df['label']==0])}")
    print(f"  Fake news (1): {len(df[df['label']==1])}")
    
    # Extract texts and labels
    texts = df['text'].astype(str).tolist()
    labels = df['label'].tolist()
    
    # Split into train and test
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        texts, labels,
        test_size=CONFIG['test_size'],
        random_state=42,
        stratify=labels
    )
    
    print(f"\n✅ Split complete:")
    print(f"  Training samples: {len(train_texts)}")
    print(f"  Test samples: {len(test_texts)}")
    
    return train_texts, train_labels, test_texts, test_labels

def tokenize_data(texts, tokenizer, max_length):
    """Tokenize texts"""
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

def train_epoch(model, dataloader, optimizer, scheduler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(dataloader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    """Evaluate the model"""
    model.eval()
    predictions = []
    true_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=-1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average='binary'
    )
    cm = confusion_matrix(true_labels, predictions)
    
    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'predictions': predictions,
        'true_labels': true_labels
    }

def test_model_sanity(model, tokenizer, device):
    """Test model with obvious examples"""
    print("\n" + "="*70)
    print("🧪 SANITY CHECK - Testing with obvious examples")
    print("="*70)
    
    test_cases = [
        ("The President held a press conference today to discuss economic policy.", 0, "REAL"),
        ("Scientists at Harvard published research on climate change in Nature journal.", 0, "REAL"),
        ("BREAKING: Aliens land in Times Square demanding pizza! You won't believe what happens next!", 1, "FAKE"),
        ("Doctors hate him! This one weird trick will make you immortal!", 1, "FAKE"),
    ]
    
    model.eval()
    correct = 0
    
    for text, expected_label, expected_name in test_cases:
        inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
        
        pred = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred].item()
        
        status = "✅" if pred == expected_label else "❌"
        correct += (pred == expected_label)
        
        print(f"\n{status} Expected: {expected_name} (label={expected_label}) | Predicted: {'REAL' if pred==0 else 'FAKE'} (confidence: {confidence:.2%})")
        print(f"   Text: {text[:80]}...")
        print(f"   Probabilities: Real={probs[0][0]:.2%}, Fake={probs[0][1]:.2%}")
    
    print(f"\n{'='*70}")
    print(f"Sanity Check: {correct}/{len(test_cases)} correct")
    print(f"{'='*70}")
    
    return correct == len(test_cases)

def main():
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("="*70)
    print("🚀 FAKE NEWS DETECTION - MODEL TRAINING")
    print("="*70)
    print(f"\n🖥️  Device: {device}")
    
    if torch.cuda.is_available():
        print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
        print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Load data
    train_texts, train_labels, test_texts, test_labels = load_dataset()
    
    if train_texts is None:
        return
    
    # Initialize model
    print(f"\n🤖 Loading {CONFIG['model_name']}...")
    tokenizer = DistilBertTokenizer.from_pretrained(CONFIG['model_name'])
    model = DistilBertForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=2
    ).to(device)
    
    # Tokenize
    print("\n🔤 Tokenizing data...")
    train_encodings = tokenize_data(train_texts, tokenizer, CONFIG['max_length'])
    test_encodings = tokenize_data(test_texts, tokenizer, CONFIG['max_length'])
    
    # Create datasets
    train_dataset = FakeNewsDataset(train_encodings, train_labels)
    test_dataset = FakeNewsDataset(test_encodings, test_labels)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'])
    
    # Setup optimizer
    optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
    total_steps = len(train_loader) * CONFIG['epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=CONFIG['warmup_steps'],
        num_training_steps=total_steps
    )
    
    # Training loop
    print(f"\n{'='*70}")
    print(f"🏋️  TRAINING FOR {CONFIG['epochs']} EPOCHS")
    print(f"{'='*70}")
    
    best_f1 = 0
    
    for epoch in range(CONFIG['epochs']):
        print(f"\n{'='*70}")
        print(f"📅 Epoch {epoch + 1}/{CONFIG['epochs']}")
        print(f"{'='*70}")
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        print(f"\n📉 Train Loss: {train_loss:.4f}")
        
        # Evaluate
        metrics = evaluate(model, test_loader, device)
        
        print(f"\n📊 Validation Metrics:")
        print(f"  Loss: {metrics['loss']:.4f}")
        print(f"  Accuracy: {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall: {metrics['recall']:.4f}")
        print(f"  F1 Score: {metrics['f1']:.4f}")
        
        print(f"\n📈 Confusion Matrix:")
        print(f"              Predicted")
        print(f"              Real  Fake")
        print(f"  Actual Real  {metrics['confusion_matrix'][0][0]:4d}  {metrics['confusion_matrix'][0][1]:4d}")
        print(f"  Actual Fake  {metrics['confusion_matrix'][1][0]:4d}  {metrics['confusion_matrix'][1][1]:4d}")
        
        # Save best model
        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            print(f"\n💾 New best model! (F1: {best_f1:.4f}) - Saving...")
            os.makedirs(CONFIG['save_path'], exist_ok=True)
            model.save_pretrained(CONFIG['save_path'])
            tokenizer.save_pretrained(CONFIG['save_path'])
            
            # Save config
            with open(f"{CONFIG['save_path']}/training_config.json", 'w') as f:
                json.dump({
                    'label_mapping': {'0': 'REAL', '1': 'FAKE'},
                    'best_f1': best_f1,
                    'best_accuracy': metrics['accuracy'],
                    'epochs': CONFIG['epochs']
                }, f, indent=2)
    
    # Final sanity check
    print("\n" + "="*70)
    print("🎓 TRAINING COMPLETE - Running final sanity check...")
    print("="*70)
    
    sanity_passed = test_model_sanity(model, tokenizer, device)
    
    print("\n" + "="*70)
    print("✅ TRAINING SUMMARY")
    print("="*70)
    print(f"  Best F1 Score: {best_f1:.4f}")
    print(f"  Model saved to: {CONFIG['save_path']}")
    print(f"  Sanity check: {'✅ PASSED' if sanity_passed else '⚠️  REVIEW NEEDED'}")
    print("\n📝 Next steps:")
    print("  1. Run: python app.py")
    print("  2. Open: http://localhost:5000")
    print("  3. Test with real and fake news!")
    print("="*70)

if __name__ == "__main__":
    main()

🚀 FAKE NEWS DETECTION - MODEL TRAINING

🖥️  Device: cpu
📥 Loading dataset...

📊 Dataset loaded:
  Total samples: 44889
  Real news (0): 21416
  Fake news (1): 23473

✅ Split complete:
  Training samples: 35911
  Test samples: 8978

🤖 Loading distilbert-base-uncased...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🔤 Tokenizing data...

🏋️  TRAINING FOR 1 EPOCHS

📅 Epoch 1/1


/tmp/ipykernel_28588/3230029983.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
Training: 100%|██████████████| 2245/2245 [2:50:12<00:00,  4.55s/it, loss=0.0001]



📉 Train Loss: 0.0401


Evaluating: 100%|█████████████████████████████| 562/562 [06:21<00:00,  1.47it/s]



📊 Validation Metrics:
  Loss: 0.0016
  Accuracy: 0.9997 (99.97%)
  Precision: 0.9994
  Recall: 1.0000
  F1 Score: 0.9997

📈 Confusion Matrix:
              Predicted
              Real  Fake
  Actual Real  4280     3
  Actual Fake     0  4695

💾 New best model! (F1: 0.9997) - Saving...

🎓 TRAINING COMPLETE - Running final sanity check...

🧪 SANITY CHECK - Testing with obvious examples

❌ Expected: REAL (label=0) | Predicted: FAKE (confidence: 54.84%)
   Text: The President held a press conference today to discuss economic policy....
   Probabilities: Real=45.16%, Fake=54.84%

✅ Expected: REAL (label=0) | Predicted: REAL (confidence: 60.27%)
   Text: Scientists at Harvard published research on climate change in Nature journal....
   Probabilities: Real=60.27%, Fake=39.73%

✅ Expected: FAKE (label=1) | Predicted: FAKE (confidence: 99.98%)
   Text: BREAKING: Aliens land in Times Square demanding pizza! You won't believe what ha...
   Probabilities: Real=0.02%, Fake=99.98%

✅ Expected: FA

In [ ]:
from flask import Flask, request, jsonify, render_template_string
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse
import warnings
warnings.filterwarnings('ignore')

app = Flask(__name__)

# Global variables
model = None
tokenizer = None
device = None

def load_model(model_path='./fake_news_model'):
    """Load the trained model"""
    global model, tokenizer, device
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🖥️  Loading model on {device}...")
    
    try:
        tokenizer = DistilBertTokenizer.from_pretrained(model_path)
        model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
        model.eval()
        
        print("✅ Model loaded successfully!")
        print("📋 Label mapping: 0=REAL, 1=FAKE")
        
        # Quick test
        test_text = "The stock market reached new highs today."
        inputs = tokenizer(test_text, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
        print(f"🧪 Test prediction: Real={probs[0][0]:.2%}, Fake={probs[0][1]:.2%}")
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("Please make sure you've run: python train_model.py first")
        raise

def scrape_article(url):
    """Scrape article text and metadata from URL"""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove unwanted elements
        for script in soup(["script", "style", "nav", "footer", "header", "aside"]):
            script.decompose()
        
        # Extract title
        title = ""
        if soup.find('h1'):
            title = soup.find('h1').get_text()
        elif soup.find('title'):
            title = soup.find('title').get_text()
        
        # Extract article text
        article_text = ""
        
        # Try common article containers
        article_containers = soup.find_all(['article', 'div'], class_=re.compile(r'article|content|post|story|body'))
        if article_containers:
            for container in article_containers:
                paragraphs = container.find_all('p')
                article_text += ' '.join([p.get_text() for p in paragraphs])
        else:
            paragraphs = soup.find_all('p')
            article_text = ' '.join([p.get_text() for p in paragraphs])
        
        # Clean text
        article_text = re.sub(r'\s+', ' ', article_text).strip()
        
        # Extract metadata
        author = ""
        date = ""
        source = urlparse(url).netloc
        
        author_meta = soup.find('meta', attrs={'name': re.compile(r'author', re.I)})
        if author_meta and author_meta.get('content'):
            author = author_meta['content']
        
        date_meta = soup.find('meta', attrs={'property': re.compile(r'published_time|date', re.I)})
        if date_meta and date_meta.get('content'):
            date = date_meta['content']
        
        full_text = f"{title} {article_text}"
        
        return {
            'success': True,
            'text': full_text[:5000],
            'metadata': {
                'title': title[:200] if title else "N/A",
                'author': author if author else "Unknown",
                'date': date if date else "Unknown",
                'source': source
            }
        }
    
    except Exception as e:
        return {
            'success': False,
            'error': str(e)
        }

def predict_fake_news(text):
    """
    Predict if news is fake or real
    Returns proper classification with confidence
    Label 0 = REAL, Label 1 = FAKE
    """
    if not text or len(text.strip()) < 10:
        return {
            'error': 'Text too short. Please provide at least 10 characters.'
        }
    
    # Tokenize
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    ).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
    
    # Get prediction (0=REAL, 1=FAKE)
    prediction = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][prediction].item()
    
    # Get individual probabilities
    real_prob = probabilities[0][0].item()  # Label 0 = REAL
    fake_prob = probabilities[0][1].item()  # Label 1 = FAKE
    
    return {
        'prediction': 'FAKE' if prediction == 1 else 'REAL',
        'confidence': float(confidence),
        'fake_probability': float(fake_prob),
        'real_probability': float(real_prob),
        'label': prediction
    }

HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Fake News Detector </title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }
        
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }
        
        .container {
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            max-width: 900px;
            width: 100%;
            padding: 40px;
            animation: fadeIn 0.5s ease-in;
        }
        
        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }
        
        h1 {
            text-align: center;
            color: #333;
            margin-bottom: 10px;
            font-size: 2.5em;
        }
        
        .subtitle {
            text-align: center;
            color: #666;
            margin-bottom: 30px;
            font-size: 1.1em;
        }
        
        .tabs {
            display: flex;
            gap: 10px;
            margin-bottom: 20px;
        }
        
        .tab {
            flex: 1;
            padding: 15px;
            background: #f0f0f0;
            border: none;
            border-radius: 10px;
            cursor: pointer;
            font-size: 16px;
            font-weight: 600;
            transition: all 0.3s;
        }
        
        .tab.active {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
        }
        
        .tab:hover:not(.active) {
            background: #e0e0e0;
        }
        
        .input-group {
            display: none;
        }
        
        .input-group.active {
            display: block;
        }
        
        textarea, input[type="url"] {
            width: 100%;
            padding: 15px;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            font-size: 16px;
            font-family: inherit;
            resize: vertical;
            transition: border-color 0.3s;
        }
        
        textarea {
            min-height: 150px;
        }
        
        textarea:focus, input[type="url"]:focus {
            outline: none;
            border-color: #667eea;
        }
        
        button.analyze-btn {
            width: 100%;
            padding: 15px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 18px;
            font-weight: 600;
            cursor: pointer;
            margin-top: 20px;
            transition: transform 0.2s, box-shadow 0.2s;
        }
        
        button.analyze-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(102, 126, 234, 0.4);
        }
        
        button.analyze-btn:active {
            transform: translateY(0);
        }
        
        button.analyze-btn:disabled {
            opacity: 0.6;
            cursor: not-allowed;
        }
        
        .loading {
            display: none;
            text-align: center;
            margin: 20px 0;
        }
        
        .loading.active {
            display: block;
        }
        
        .spinner {
            border: 4px solid #f3f3f3;
            border-top: 4px solid #667eea;
            border-radius: 50%;
            width: 40px;
            height: 40px;
            animation: spin 1s linear infinite;
            margin: 0 auto;
        }
        
        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }
        
        .result {
            display: none;
            margin-top: 30px;
            padding: 30px;
            border-radius: 15px;
            animation: slideIn 0.5s ease-out;
        }
        
        @keyframes slideIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }
        
        .result.active {
            display: block;
        }
        
        .result.fake {
            background: linear-gradient(135deg, #ff6b6b 0%, #ee5a6f 100%);
            color: white;
        }
        
        .result.real {
            background: linear-gradient(135deg, #51cf66 0%, #37b24d 100%);
            color: white;
        }
        
        .result-header {
            display: flex;
            align-items: center;
            gap: 15px;
            margin-bottom: 20px;
        }
        
        .result-icon {
            font-size: 3em;
        }
        
        .result-text h2 {
            font-size: 2em;
            margin-bottom: 5px;
        }
        
        .confidence-bar {
            background: rgba(255,255,255,0.3);
            border-radius: 10px;
            height: 30px;
            margin: 20px 0;
            overflow: hidden;
        }
        
        .confidence-fill {
            height: 100%;
            background: white;
            border-radius: 10px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 600;
            transition: width 1s ease-out;
            color: #333;
        }
        
        .probabilities {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 15px;
            margin-top: 20px;
        }
        
        .prob-card {
            background: rgba(255,255,255,0.2);
            padding: 15px;
            border-radius: 10px;
            text-align: center;
        }
        
        .prob-card h3 {
            font-size: 0.9em;
            margin-bottom: 10px;
            opacity: 0.9;
        }
        
        .prob-card .value {
            font-size: 2em;
            font-weight: 700;
        }
        
        .metadata {
            margin-top: 20px;
            padding: 15px;
            background: rgba(255,255,255,0.2);
            border-radius: 10px;
        }
        
        .metadata h3 {
            margin-bottom: 10px;
            font-size: 1.2em;
        }
        
        .metadata p {
            margin: 5px 0;
            opacity: 0.9;
        }
        
        .error {
            background: #ff6b6b;
            color: white;
            padding: 15px;
            border-radius: 10px;
            margin-top: 20px;
            display: none;
        }
        
        .error.active {
            display: block;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🔍 Fake News Detector</h1>
        <p class="subtitle">Powered by AI • Analyze text or URLs instantly</p>
        
        <div class="tabs">
            <button class="tab active" onclick="switchTab('text')">📝 Text Analysis</button>
            <button class="tab" onclick="switchTab('url')">🔗 URL Analysis</button>
        </div>
        
        <div id="textInput" class="input-group active">
            <textarea id="newsText" placeholder="Paste your news article here..."></textarea>
            <button class="analyze-btn" onclick="analyzeText()">Analyze Text</button>
        </div>
        
        <div id="urlInput" class="input-group">
            <input type="url" id="newsUrl" placeholder="https://example.com/news-article">
            <button class="analyze-btn" onclick="analyzeUrl()">Analyze URL</button>
        </div>
        
        <div class="loading">
            <div class="spinner"></div>
            <p style="margin-top: 10px; color: #666;">Analyzing...</p>
        </div>
        
        <div class="error"></div>
        
        <div class="result"></div>
    </div>
    
    <script>
        function switchTab(tab) {
            document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
            document.querySelectorAll('.input-group').forEach(g => g.classList.remove('active'));
            
            if (tab === 'text') {
                document.querySelectorAll('.tab')[0].classList.add('active');
                document.getElementById('textInput').classList.add('active');
            } else {
                document.querySelectorAll('.tab')[1].classList.add('active');
                document.getElementById('urlInput').classList.add('active');
            }
            
            hideResult();
        }
        
        function showLoading() {
            document.querySelector('.loading').classList.add('active');
            document.querySelector('.error').classList.remove('active');
            document.querySelectorAll('.analyze-btn').forEach(btn => btn.disabled = true);
        }
        
        function hideLoading() {
            document.querySelector('.loading').classList.remove('active');
            document.querySelectorAll('.analyze-btn').forEach(btn => btn.disabled = false);
        }
        
        function hideResult() {
            document.querySelector('.result').classList.remove('active');
            document.querySelector('.error').classList.remove('active');
        }
        
        function showError(message) {
            const errorDiv = document.querySelector('.error');
            errorDiv.textContent = '❌ ' + message;
            errorDiv.classList.add('active');
        }
        
        function displayResult(data, metadata = null) {
            const resultDiv = document.querySelector('.result');
            const isFake = data.prediction === 'FAKE';
            
            resultDiv.className = 'result active ' + (isFake ? 'fake' : 'real');
            
            let metadataHtml = '';
            if (metadata) {
                metadataHtml = `
                    <div class="metadata">
                        <h3>📄 Article Information</h3>
                        ${metadata.title ? `<p><strong>Title:</strong> ${metadata.title}</p>` : ''}
                        ${metadata.author ? `<p><strong>Author:</strong> ${metadata.author}</p>` : ''}
                        ${metadata.date ? `<p><strong>Date:</strong> ${metadata.date}</p>` : ''}
                        ${metadata.source ? `<p><strong>Source:</strong> ${metadata.source}</p>` : ''}
                    </div>
                `;
            }
            
            resultDiv.innerHTML = `
                <div class="result-header">
                    <div class="result-icon">${isFake ? '⚠️' : '✅'}</div>
                    <div class="result-text">
                        <h2>${data.prediction} NEWS</h2>
                        <p>Confidence: ${(data.confidence * 100).toFixed(1)}%</p>
                    </div>
                </div>
                
                <div class="confidence-bar">
                    <div class="confidence-fill" style="width: ${data.confidence * 100}%">
                        ${(data.confidence * 100).toFixed(1)}%
                    </div>
                </div>
                
                <div class="probabilities">
                    <div class="prob-card">
                        <h3>Real News</h3>
                        <div class="value">${(data.real_probability * 100).toFixed(1)}%</div>
                    </div>
                    <div class="prob-card">
                        <h3>Fake News</h3>
                        <div class="value">${(data.fake_probability * 100).toFixed(1)}%</div>
                    </div>
                </div>
                
                ${metadataHtml}
            `;
        }
        
        async function analyzeText() {
            const text = document.getElementById('newsText').value.trim();
            
            if (!text) {
                showError('Please enter some text to analyze');
                return;
            }
            
            showLoading();
            hideResult();
            
            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ text: text })
                });
                
                const data = await response.json();
                hideLoading();
                
                if (data.error) {
                    showError(data.error);
                } else {
                    displayResult(data);
                }
            } catch (error) {
                hideLoading();
                showError('Failed to analyze text. Please try again.');
            }
        }
        
        async function analyzeUrl() {
            const url = document.getElementById('newsUrl').value.trim();
            
            if (!url) {
                showError('Please enter a URL to analyze');
                return;
            }
            
            showLoading();
            hideResult();
            
            try {
                const response = await fetch('/predict_url', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ url: url })
                });
                
                const data = await response.json();
                hideLoading();
                
                if (data.error) {
                    showError(data.error);
                } else {
                    displayResult(data, data.metadata);
                }
            } catch (error) {
                hideLoading();
                showError('Failed to analyze URL. Please try again.');
            }
        }
        
        document.getElementById('newsText').addEventListener('keydown', function(e) {
            if (e.ctrlKey && e.key === 'Enter') {
                analyzeText();
            }
        });
        
        document.getElementById('newsUrl').addEventListener('keydown', function(e) {
            if (e.key === 'Enter') {
                analyzeUrl();
            }
        });
    </script>
</body>
</html>
'''

@app.route('/')
def home():
    return render_template_string(HTML_TEMPLATE)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        text = data.get('text', '')
        
        result = predict_fake_news(text)
        return jsonify(result)
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/predict_url', methods=['POST'])
def predict_url():
    try:
        data = request.json
        url = data.get('url', '')
        
        # Scrape article
        scraped = scrape_article(url)
        
        if not scraped['success']:
            return jsonify({'error': f'Failed to scrape article: {scraped["error"]}'}), 400
        
        # Predict
        result = predict_fake_news(scraped['text'])
        
        if 'error' not in result:
            result['metadata'] = scraped['metadata']
        
        return jsonify(result)
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    # Load model on startup
    load_model()
    
    # Run app
    print("\n Starting Fake News Detector Web App...")
    print(" Access at: http://localhost:5000")
    
    app.run(debug=False, host='0.0.0.0', port=5000, use_reloader=False)

🖥️  Loading model on cpu...
✅ Model loaded successfully!
📋 Label mapping: 0=REAL, 1=FAKE
🧪 Test prediction: Real=3.79%, Fake=96.21%

 Starting Fake News Detector Web App...
 Access at: http://localhost:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.29.173:5000
Press CTRL+C to quit
192.168.29.173 - - [16/Nov/2025 20:36:00] "GET / HTTP/1.1" 200 -
192.168.29.173 - - [16/Nov/2025 20:36:00] "GET /favicon.ico HTTP/1.1" 404 -
192.168.29.173 - - [16/Nov/2025 20:36:36] "POST /predict HTTP/1.1" 200 -
192.168.29.173 - - [16/Nov/2025 20:39:03] "POST /predict HTTP/1.1" 200 -
192.168.29.173 - - [16/Nov/2025 20:40:08] "POST /predict_url HTTP/1.1" 400 -
192.168.29.173 - - [16/Nov/2025 20:41:35] "POST /predict_url HTTP/1.1" 200 -
192.168.29.173 - - [16/Nov/2025 20:43:25] "POST /predict_url HTTP/1.1" 200 -
